<!-- SPDX-FileCopyrightText: Copyright (c) 2026, NVIDIA CORPORATION & AFFILIATES. All rights reserved. -->
<!-- SPDX-License-Identifier: Apache-2.0 -->
# Second-look video analysis

Run a first read and one narrower read through `vss vlm run`, which calls core `second_look()`.
The narrower answer is selected when available; a failed optional read retains the first answer.
This is an explicit window workflow, not confidence gating or automatic event localization.
Prefer moment-specific questions in long videos when a timestamp or grounded observation locates the event.
Skip re-look when the moment is unknown, the whole recording is needed, or adequate evidence is already visible but the model cannot interpret it. Low confidence alone is not a trigger.
The selected narrower answer may regress; both answers are retained for review.

Use the CLI installed from this checkout with an already configured VSS deployment
([setup](../../../AGENTS.md)). The sensor must already be registered.
Choose both windows from its recorded timeline (`vss vios timeline --sensor NAME`).
Both must fit inside one recorded segment, with the second strictly narrower than the first.
Use seconds from the start for a file-backed recording, or ISO-8601 UTC timestamps for a camera.
The existing configured model and sampling policy apply to both reads.
A fixed frame count on a shorter clip gives denser temporal sampling; unchanged FPS gives fewer frames.
Sampling controls depend on the backend; the plain OpenAI backend uses server defaults.


In [ ]:
SENSOR = "dock_cam"
FIRST_START, FIRST_END = "0", "30"
RELOOK_START, RELOOK_END = "12", "16"
QUESTION = "Was the worker wearing a helmet?"


In [ ]:
import json
import subprocess

completed = subprocess.run(
    [
        "vss", "vlm", "run", "--sensor", SENSOR,
        "--start-time", FIRST_START, "--end-time", FIRST_END,
        "--relook-start-time", RELOOK_START, "--relook-end-time", RELOOK_END,
        "--prompt", QUESTION,
    ],
    capture_output=True,
    text=True,
)
if completed.stderr:
    print(completed.stderr)
if completed.returncode not in (0, 6):
    raise RuntimeError(f"VSS exited {completed.returncode}; see the diagnostic above.")

# The first JSON object is the result; a completion marker follows it.
result, _ = json.JSONDecoder().raw_decode(completed.stdout.lstrip())
print("Answer:", result["answer"])
print(json.dumps(result["second_look"], indent=2))


Review the two answers and windows, call count, job IDs and `fallback_reason` above.
Exit 6 means a partial outcome: retain the answer and report the stated limitation.
Do not repeat the command to repair a failed optional read. Remove the `--relook-*`
flags for the existing single-read behavior.
